# 03 — Parâmetros Morfológicos e Térmicos

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/general/03_morphological_parameters.pt.ipynb)

**Objetivo de aprendizagem**: ir além do *mapa de classes* LCZ e explorar os 34 parâmetros morfológicos e térmicos quantitativos que Stewart & Oke (2012) associam a cada classe LCZ — fator de visão do céu, razão de aspecto, frações de superfície, rugosidade, admitância, albedo e calor antropogênico — aprendendo a extrair, selecionar, vetorizar e visualizar esses dados com `lcz_get_parameters` e `lcz_plot_parameters`.

## Por que parâmetros, e não apenas classes

O esquema de Zonas Climáticas Locais (Stewart & Oke, 2012, *Bulletin of the American Meteorological Society*) costuma ser apresentado como uma *classificação* — 17 categorias discretas, de "Compacto de alta densidade" (LCZ 1) a "Água" (LCZ 17). Mas a classificação em si é apenas metade do valor do arcabouço. O que torna o LCZ genuinamente útil para a climatologia urbana é que **cada classe carrega uma faixa padronizada de parâmetros físicos, derivada da literatura**, que controlam o balanço de energia de superfície e o clima próximo à superfície: quanto céu um cânion de rua consegue ver (fator de visão do céu, SVF), quão altos são os edifícios em relação ao espaço entre eles (razão de aspecto / altura-largura, aproximadamente o que a banda bruta `aspect_mean` representa aqui), quanto do solo está construído, pavimentado ou vegetado (frações de superfície construída/impermeável/permeável/arbórea — BSF/ISF/PSF/TSF), quão rugosa é a superfície do ponto de vista aerodinâmico (classe de rugosidade e `z0`, o comprimento de rugosidade), quão rápido o material da superfície armazena e libera calor (admitância térmica de superfície, SAD), quanta radiação de onda curta ela reflete (albedo de superfície, SAL) e quanto calor residual a atividade humana injeta diretamente na camada limite urbana (fluxo de calor antropogênico, AH).

Essa é exatamente a razão pela qual o LCZ superou o antigo paradigma "urbano vs. rural" da ilha de calor: uma zona "Compacto de média densidade" em São Paulo e outra em Berlim terão propriedades radiativas e aerodinâmicas *sistematicamente semelhantes*, mesmo parecendo completamente diferentes do ponto de vista administrativo, porque a classificação é construída diretamente a partir da morfologia, não de rótulos de uso do solo. Essa portabilidade é o que torna os mapas LCZ utilizáveis como **entradas diretas para modelos climáticos urbanos de micro e mesoescala** — as faixas de parâmetros apresentadas aqui são exatamente o tipo de tabela de referência que os modelos de dossel urbano do WRF (UCM de camada única ou múltipla, BEP/BEM) e outros esquemas de meso/microescala precisam para inicializar a física dependente da morfologia sem levantamentos específicos de cada edificação. Um modelador que tem um mapa LCZ de uma região já possui, na prática, valores de primeira estimativa para rugosidade de superfície, admitância térmica e albedo — exatamente os campos que este notebook extrai.

O `LCZ4py` implementa essa tabela de parâmetros como uma consulta (*lookup*) rápida em NumPy (veja `lcz_get_parameters`), seguindo o mesmo desenho conceitual do pacote R `LCZ4r` (Anjos, M. et al., 2025, *Scientific Reports*, https://www.nature.com/articles/s41598-025-92000-0), reimplementado aqui com indexação vetorizada 30–100x mais rápida que a abordagem original baseada em `dplyr::inner_join`. Cada uma das 17 classes mapeia para um trio `min`/`max`/`mean` em 11 famílias de parâmetros (SVF, AR, BSF, ISF, PSF, TSF, HRE — altura dos elementos de rugosidade, TRC — classe de rugosidade do terreno, SAD, SAL, AH) mais um único valor de comprimento de rugosidade (`z0`), totalizando **34 parâmetros reais** (além do próprio código de classe LCZ como banda de controle) — pixel a pixel, calculados inteiramente a partir do mapa de classes já baixado com `lcz_get_map`.

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## 1. Obter um mapa de classes LCZ

Usamos **Juiz de Fora** (Liechtenstein) como cidade de referência ao longo de todo o notebook — pequena o suficiente para que o download do mapa, a extração de parâmetros e todos os gráficos abaixo rodem em segundos, e diversa o suficiente (contém várias classes urbanas e naturais/água) para tornar as comparações significativas.

In [2]:
from LCZ4py import lcz_get_map

map_path = lcz_get_map(city="Juiz de Fora")
print(map_path)

06:27:24 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


/Users/co2map/.lcz4r_cache/clipped_a8db9faa23e32c3b.tif


## 2. Extrair a pilha completa de 34 parâmetros — `lcz_get_parameters`

`lcz_get_parameters(x, istack=True)` mapeia o raster de classes LCZ para um array `(35, H, W)` em uma única passagem vetorizada com NumPy: a banda 0 é o próprio código de classe LCZ (mantido para controle/junções), e as demais 34 bandas são as variantes `min`, `max` e `mean` das famílias de parâmetros descritas acima, além do comprimento de rugosidade `z0`. Com `isave=True`, também é escrito um GeoTIFF multibanda (`LCZ4r_output/lcz_par_stack.tif`) com o nome de cada banda armazenado como sua descrição GDAL — é isso que permite que `lcz_plot_parameters` (e, mais adiante, `lcz_cal_indices` internamente) endereçe bandas pelo nome em vez da posição, e é esse arquivo que reaproveitaremos em todos os gráficos abaixo.

Retorna um `LCZStackResult` com `.path` (se salvo), `.array` (a pilha em memória), `.gdf`/`.geoarrow_table` (não usados nesta chamada — são preenchidos pelo caminho `ishp=True` abaixo).

In [3]:
from LCZ4py import lcz_get_parameters

stack_result = lcz_get_parameters(map_path, istack=True, isave=True, lang="pt")
print("Stack shape (bands, height, width):", stack_result.array.shape)
print("Saved GeoTIFF:", stack_result.path)

06:27:24 - rasterio._env - WARNING - CPLE_IllegalArg in lcz_par_stack.tif: BLOCKXSIZE can only be used with TILED=YES


06:27:24 - LCZ4py.general.lcz_get_parameters - INFO - Saved: /Users/co2map/Documents/lcz4r_python/notebooks/general/LCZ4r_output/lcz_par_stack.tif


Stack shape (bands, height, width): (35, 120, 131)
Saved GeoTIFF: LCZ4r_output/lcz_par_stack.tif


**Interpretando a saída**: o formato `(35, H, W)` confirma uma banda para o código de classe + 34 parâmetros, na mesma grade de pixels do mapa LCZ de entrada. O valor de cada pixel é obtido diretamente a partir de sua classe LCZ — é um remapeamento categórico para contínuo, não uma medição independente, portanto todos os pixels que compartilham uma classe LCZ recebem valores de parâmetro idênticos (a *faixa dentro da classe* entre `min` e `max` reflete a variabilidade real entre cidades da síntese da literatura de Stewart & Oke, não variabilidade dentro deste raster específico).

## 3. Selecionar um subconjunto de parâmetros — `iselect`

A maioria dos fluxos de trabalho não precisa das 34 bandas. Passar `iselect` (um nome ou lista de nomes) retorna apenas essas bandas como um array NumPy simples — útil quando você só precisa, por exemplo, das frações médias de superfície e do calor antropogênico como rasters prontos para uso em modelos, sem pagar o custo de I/O da pilha completa.

In [4]:
selected = lcz_get_parameters(
    map_path, iselect=["svf_mean", "BSF_mean", "AH_mean"], lang="pt"
)
print("Selected shape:", selected.shape, type(selected))

Selected shape: (3, 120, 131) <class 'numpy.ndarray'>


**Interpretando a saída**: formato `(3, H, W)` — uma banda por nome solicitado, na ordem do pedido — `svf_mean` (fator de visão do céu médio), `BSF_mean` (fração média de superfície construída), `AH_mean` (fluxo médio de calor antropogênico, W/m²). Essa é a forma mais enxuta de obter exatamente os campos de que um modelo ou análise downstream precisa.

## 4. Vetorizar para uma tabela de polígonos — `ishp=True`

`ishp=True` dissolve o raster de classes em um polígono por zona LCZ contígua (via DuckDB Spatial quando disponível, com fallback para GeoPandas) e associa cada parâmetro como coluna de atributo. Este é o formato natural para fluxos de trabalho em SIG — por exemplo, entregar uma tabela de parâmetros por zona a um pré-processador do WRF, a uma camada do QGIS, ou a um GeoPackage para uma equipe de modelagem que não trabalha em Python/espaço raster.

In [5]:
gdf_result = lcz_get_parameters(map_path, ishp=True, lang="pt")
print(gdf_result.gdf.shape, "polygons x columns")
gdf_result.gdf[["lcz", "svf_mean", "BSF_mean", "AH_mean", "z0", "geometry"]].head()

06:27:24 - LCZ4py.general.lcz_get_parameters - INFO - Vectorizing LCZ classes...


06:27:24 - rasterio._env - WARNING - CPLE_AppDefined in DeprecationWarning: 'Memory' driver is deprecated since GDAL 3.11. Use 'MEM' onwards. Further messages of this type will be suppressed.


06:27:25 - LCZ4py.general.lcz_get_parameters - WARNING - DuckDB dissolve failed, falling back to GeoPandas: Not implemented Error: Data type 'geometry' not recognized


(9, 36) polygons x columns


,lcz,svf_mean,BSF_mean,AH_mean,z0,geometry
0,5,0.65,30.0,24.0,0.50,"POLYGON ((9.52124 47.14359, 9.52124 47.14269, ..."
1,6,0.75,30.0,24.0,0.50,"POLYGON ((9.50418 47.15706, 9.50418 47.15616, ..."
2,8,0.75,40.0,49.0,0.25,"MULTIPOLYGON (((9.51765 47.12292, 9.51765 47.1..."
3,9,0.85,15.0,9.0,0.50,"MULTIPOLYGON (((9.51855 47.13191, 9.51855 47.1..."
4,11,0.35,9.0,0.0,2.00,"MULTIPOLYGON (((9.60928 47.09867, 9.60928 47.0..."


**Interpretando a saída**: uma linha por zona LCZ presente na área de Juiz de Fora, com a tabela completa de parâmetros associada pela classe (`lcz`) mais uma coluna `geometry`. Note que `z0` e todos os campos `min`/`max`/`mean` são *constantes por classe* aqui, não estimativas por polígono — a geometria do polígono é o que varia (forma e área), os atributos são os valores de referência derivados da literatura para aquela classe.

## 5. Visualizar padrões espaciais — `chart_type="map"`

`lcz_plot_parameters` renderiza qualquer banda de parâmetro como um mapa de calor interativo em Plotly. Usamos o caminho da pilha salva na etapa 2 (que carrega os nomes de banda como descrições GDAL) e selecionamos com `iselect` três parâmetros médios representativos: fator de visão do céu médio (`svf_mean`), fração média de superfície construída (`BSF_mean`) e calor antropogênico médio (`AH_mean`).

## Mapas de parâmetros prontos para publicação com `style`

As funções de plotagem agora aceitam `style="default"`, `style="nature"`, `style="science"` e `style="generic_bw"`. Os presets de revista ajustam a família de fontes, o tamanho físico em milímetros, o DPI (300 para os estilos voltados à publicação) e as cores para que mapas de parâmetros possam ser exportados como figuras prontas para manuscrito sem esforço extra de estilização.

Quando a visualização do parâmetro é um mapa, os mesmos auxiliares cartográficos usados nas outras funções também estão disponíveis: `add_scalebar=True` e `add_north_arrow=True`.

Um exemplo mínimo pronto para publicação:

- `style="nature"` para o painel de mapa.
- `isave=True` com `save_extension="png"` ou `"pdf"` para gravar a figura.


In [ ]:
from LCZ4py.general.lcz_plot_parameters import lcz_plot_parameters

figs_pub = lcz_plot_parameters(
    result_stack,
    all_params=False,
    iselect=["svf_mean", "BSF_mean", "AH_mean"],
    chart_type="map",
    style="nature",
    isave=True,
    save_extension="png",
    add_scalebar=True,
    add_north_arrow=True,
)
figs_pub


In [6]:
from LCZ4py import lcz_plot_parameters

map_figs = lcz_plot_parameters(
    stack_result.path, iselect=["svf_mean", "BSF_mean", "AH_mean"],
    chart_type="map", lang="pt",
)
for fig in map_figs:
    fig.show()

**Interpretando a saída**: três mapas de calor sobre a área exata do mapa LCZ. `svf_mean` deve ser *menor* nas classes compactas/densas (cânions de rua estreitos veem pouco céu) e maior sobre áreas abertas/naturais; `BSF_mean` e `AH_mean` devem acompanhar-se mutuamente e atingir picos onde estão as classes mais densamente construídas do mapa (indústria/zonas compactas), caindo para ~0 sobre vegetação e água. Se os três mapas não mostrarem esse padrão inverso entre SVF e BSF/AH, isso é um sinal de que algo está errado no raster de classes de entrada, não na tabela de parâmetros.

## 6. Relações entre parâmetros — `chart_type="scatter"`

Para valores de `chart_type` diferentes de `"map"`, `lcz_plot_parameters` delega para `lcz_cal_indices` (o mesmo motor de estatísticas por classe usado para índices espectrais) — ele funciona com *qualquer* GeoTIFF multibanda com bandas nomeadas, incluindo parâmetros morfológicos, já que a lógica de agregação nunca assume nada sobre o que as bandas representam fisicamente. Isso requer `lcz_map` (o raster de classes, para atribuir a classe LCZ de cada pixel para coloração/estatísticas).

`"scatter"` plota cada pixel de `index_x` contra `index_y`, colorido pela classe LCZ usando a paleta oficial LCZ — aqui, fração média de superfície construída vs. fator de visão do céu médio.

In [7]:
scatter_fig = lcz_plot_parameters(
    stack_result.path, lcz_map=map_path, chart_type="scatter",
    index_x="BSF_mean", index_y="svf_mean", lang="pt",
)
scatter_fig.show()

**Interpretando a saída**: como ambos os eixos são constantes de classe (não medições por pixel), cada classe LCZ colapsa para um único ponto (ou um pequeno agrupamento onde o raster mistura pixels de borda/resoluções próximas), em vez de uma nuvem dispersa — e esse é justamente o objetivo: torna explícita e comparável classe a classe a relação inversa, já conhecida de Stewart & Oke, entre densidade construtiva e céu aberto. Classes densas/compactas ficam no canto de SVF baixo/BSF alto; classes abertas, naturais e de água ficam no canto oposto.

## 7. Formato do perfil por classe — `chart_type="radar"`

`"radar"` desenha um traço polar por classe LCZ ao longo de vários parâmetros solicitados, cada eixo normalizado por min-max (`normalize_radar=True`, padrão) para que parâmetros com faixas naturais muito diferentes (0–1 do SVF vs. dezenas a centenas de W/m² do AH) permaneçam visualmente comparáveis no mesmo gráfico. Esta é a forma mais clara de comparar a *assinatura morfológica geral* de uma classe em relação a outra — não um parâmetro de cada vez, mas o formato completo.

In [8]:
profile_params = [
    "svf_mean", "aspect_mean", "BSF_mean", "ISF_mean",
    "PSF_mean", "TSF_mean", "AH_mean",
]

radar_fig = lcz_plot_parameters(
    stack_result.path, iselect=profile_params, lcz_map=map_path,
    chart_type="radar", lang="pt",
)
radar_fig.show()

**Interpretando a saída**: `aspect_mean` aqui é a banda da razão de aspecto (altura/largura) média (rotulada `aspect_mean` no raster, correspondendo ao "AR" de Stewart & Oke). Classes urbanas compactas/densas devem traçar um perfil inclinado para `BSF_mean`/`ISF_mean`/`aspect_mean`/`AH_mean` altos e `svf_mean`/`TSF_mean`/`PSF_mean` baixos; classes naturais traçam algo próximo da imagem espelhada. Uma classe cujo polígono parece "pontudo" em um eixo em relação às vizinhas é a que tem a morfologia mais distintiva para aquele parâmetro — útil para identificar quais classes uma dada intervenção urbana (por exemplo, aumentar a cobertura arbórea) mais deslocaria.

## 8. Redundância entre parâmetros — `chart_type="correlation"`

`"correlation"` calcula a matriz de correlação de Pearson entre os parâmetros solicitados sobre todos os pixels LCZ válidos. Com 34 parâmetros construídos a partir de apenas 11 famílias-base (cada uma dividida em min/max/mean), é esperado bastante redundância — este gráfico a torna explícita, o que importa se você for alimentar esses campos em um modelo estatístico ou de aprendizado de máquina, onde preditores quase duplicados apenas adicionam colinearidade sem adicionar informação.

In [9]:
corr_fig = lcz_plot_parameters(
    stack_result.path, iselect=profile_params, lcz_map=map_path,
    chart_type="correlation", lang="pt",
)
corr_fig.show()

**Interpretando a saída**: `BSF_mean` e `ISF_mean` (frações de superfície construída e impermeável) tipicamente mostram forte correlação positiva — ambas medem "o quanto esta classe está pavimentada," só que por ângulos ligeiramente diferentes — enquanto `svf_mean` costuma anti-correlacionar fortemente com `aspect_mean` e `BSF_mean` (mais volume construído bloqueia mais céu, por construção). `TSF_mean`/`PSF_mean` (frações arbórea/permeável) tendem a anti-correlacionar com o trio construído. Se você estivesse selecionando um conjunto mínimo de variáveis para, digamos, um modelo de interpolação por aprendizado de máquina (veja `lcz_interp_map_plus` na série `local/`), esta matriz é exatamente como você decidiria quais parâmetros são redundantes e seguros de descartar.

## Recapitulação e próximos passos

Neste notebook você foi além do mapa de *classes* LCZ e explorou a assinatura morfológica/térmica completa de **34 parâmetros** usando `lcz_get_parameters` — extraindo a pilha completa, um subconjunto enxuto (`iselect`) e uma tabela de polígonos pronta para SIG (`ishp=True`) — e então explorou essa assinatura de quatro formas com `lcz_plot_parameters`: padrão espacial (`"map"`), relações par a par (`"scatter"`), formato do perfil por classe (`"radar"`) e redundância entre parâmetros (`"correlation"`).

- Anterior: [`02_visualization_area_stats`](02_visualization_area_stats.pt.ipynb) — visualização e estatísticas de área.
- Próximo: [`04_remote_sensing_lst_pc`](04_remote_sensing_lst_pc.pt.ipynb) — trazendo observações reais de sensoriamento remoto (Temperatura de Superfície, Microsoft Planetary Computer) para comparar com essas estimativas de parâmetros derivadas da literatura.